# Building Your First Multi-Agent System — A Beginner's Guide (CrewAI)

**Raz Systems · Agentic AI Engineering Curriculum**

Based on: *"Building Your First Multi-Agent System: A Beginner's Guide"* (MachineLearningMastery.com, by Cornellius Yudha Wijaya)

---

**Two changes from the original article, both deliberate:**

1. **Search tool**: the original uses Brave Search via a hand-wrapped `CrewStructuredTool`. This notebook uses **Serper** (`SERPER_API_KEY`) instead, via CrewAI's own built-in `SerperDevTool` — both because Serper is the search provider already used throughout the Raz Systems curriculum, and because `CrewStructuredTool` is no longer the supported way to register custom tools in current CrewAI versions (more on this in Section 3).
2. **Topic**: instead of a generic `{topic}` placeholder, every example run below uses **Bangalore's tech industry and startup ecosystem** as the concrete topic, so you can see real agent output rather than an abstract placeholder.

## What you'll build

A **multi-agent system** — several specialized agents collaborating toward one shared goal, instead of a single agent doing everything. Per the article, an individual agent's cognitive architecture has three parts:

1. **A language model** as the central decision-making engine
2. **Tools** for the agent to interact with external systems
3. **Orchestration** that governs how the agents act

A single agent is fine for simple tasks, but as task complexity grows, the chance of any one agent failing to reach the goal increases. A **multi-agent system** addresses this by letting multiple specialized agents each handle one piece of the problem.

## Two common architectures

- **Network architecture** — agents talk directly to each other and jointly decide who acts next.
- **Supervisor architecture** — a single supervisor (manager) agent decides which agent should act next.

This notebook builds a **supervisor architecture**: one manager agent coordinates four specialized agents to produce a research report.

## The system we're building

**Goal:** produce a polished report on a topic — here, *"Bangalore's tech industry and startup ecosystem"*.

**Four specialized agents, plus one manager:**

1. **Web researcher agent** — searches the web (via Serper) for current, relevant facts about the topic
2. **Trend analyst agent** — analyzes the research and ranks the most significant trends
3. **Report writer agent** — drafts a structured report from the research + analysis
4. **Proofreader agent** — polishes the draft for grammar, coherence, and formatting

The **manager agent** supervises all four — it decides which agent acts on which task, and in what order, rather than you hardcoding the sequence yourself.

## 1. Setup

In [ ]:
!pip install -q crewai crewai-tools python-dotenv

In [1]:
import os
from dotenv import load_dotenv

from crewai import Crew, Task, Agent, Process, LLM

load_dotenv()

# Required environment variables:
#   OPENAI_API_KEY   — for the LLM powering every agent
#   SERPER_API_KEY   — for the web researcher agent's search tool (read automatically by SerperDevTool)

True

## 2. The LLM

Every agent in this system is powered by the same underlying model — CrewAI uses LiteLLM under the hood, so any model LiteLLM supports works here. The article uses GPT-4o; we keep that choice, but you can swap it for any model string LiteLLM recognizes.

In [2]:
llm = LLM(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 3. The search tool — Serper instead of Brave

The original article wraps Brave Search manually using `CrewStructuredTool`. We make two changes here:

1. **Serper instead of Brave** — same provider used throughout the rest of the Raz Systems curriculum (`google.serper.dev`).
2. **The built-in `SerperDevTool`** instead of hand-wrapping a search function — CrewAI's own tools package (`crewai-tools`) ships a ready-made Serper integration. This is also a correction versus the original article: CrewAI's tool internals have changed since that article was published, and `CrewStructuredTool` is no longer the supported way to register a custom tool on an `Agent` in current CrewAI versions. The officially supported patterns now are either subclassing `BaseTool` or using the `@tool` decorator — and since Serper is common enough that CrewAI ships first-party support for it, we use that directly rather than reinventing it.

`SerperDevTool` reads `SERPER_API_KEY` from the environment automatically — no manual request-wrapping code needed at all.

In [ ]:
!pip install -q crewai-tools

In [4]:
from crewai_tools import SerperDevTool

# Reads SERPER_API_KEY from the environment automatically
SearchTool = SerperDevTool()

print("✅ SearchTool ready:", SearchTool.name)

✅ SearchTool ready: Search the internet with Serper


## 4. Define the agents

CrewAI agents are defined with three things:

- **`role`** — who the agent is
- **`goal`** — what it's trying to achieve (note the `{topic}` placeholder — this lets us pass the actual topic in at run time, without changing any agent code)
- **`backstory`** — context that shapes *how* the agent approaches its role; more specific backstories tend to produce more specific, better-targeted output

Only the **web researcher** gets `tools=[SearchTool]` — it's the only agent that needs to reach outside the conversation to gather fresh information. The other three work purely with what's already in the conversation (the manager's task routing, and each other's outputs).

In [5]:
# Web researcher — the only agent with web search access
web_researcher_agent = Agent(
    role="Web Research Specialist",
    goal=(
        "To find the most recent, impactful, and relevant information about {topic}. This includes identifying "
        "key use cases, challenges, and statistics to provide a foundation for deeper analysis."
    ),
    backstory=(
        "You are a former investigative journalist known for your ability to uncover technology breakthroughs "
        "and market insights. With years of experience, you excel at identifying actionable data and trends."
    ),
    tools=[SearchTool],
    llm=llm,
    verbose=True,
)

In [6]:
trend_analyst_agent = Agent(
    role="Insight Synthesizer",
    goal=(
        "To analyze research findings, extract significant trends, and rank them by industry impact, growth potential, "
        "and uniqueness. Provide actionable insights for decision-makers."
    ),
    backstory=(
        "You are a seasoned strategy consultant who transitioned into {topic} analysis. With an eye for patterns, "
        "you specialize in translating raw data into clear, actionable insights."
    ),
    tools=[],
    llm=llm,
    verbose=True,
)

report_writer_agent = Agent(
    role="Narrative Architect",
    goal=(
        "To craft a detailed, professional report that communicates research findings and analysis effectively. "
        "Focus on clarity, logical flow, and engagement."
    ),
    backstory=(
        "Once a technical writer for a renowned journal, you are now dedicated to creating industry-leading reports. "
        "You blend storytelling with data to ensure your work is both informative and captivating."
    ),
    tools=[],
    llm=llm,
    verbose=True,
)

proofreader_agent = Agent(
    role="Polisher of Excellence",
    goal=(
        "To refine the report for grammatical accuracy, readability, and formatting, ensuring it meets professional "
        "publication standards."
    ),
    backstory=(
        "An award-winning editor turned proofreader, you specialize in perfecting written content. Your sharp eye for "
        "detail ensures every document is flawless."
    ),
    tools=[],
    llm=llm,
    verbose=True,
)

## 5. The manager agent

This is what makes the architecture a **supervisor** architecture rather than a network one. The manager doesn't do any research, analysis, or writing itself — its entire job is **coordinating the other four agents**: deciding which agent should act on which task, in what order, and verifying the results meet quality standards.

In [7]:
manager_agent = Agent(
    role="Workflow Maestro",
    goal=(
        "To coordinate agents, manage task dependencies, and ensure all outputs meet quality standards. Your focus "
        "is on delivering a cohesive final product through efficient task management."
    ),
    backstory=(
        "A former project manager with a passion for efficient teamwork, you ensure every process runs smoothly, "
        "overseeing tasks and verifying results."
    ),
    tools=[],
    llm=llm,
    verbose=True,
)

## 6. Define the tasks

If an **agent** is the individual, a **task** is the specific action that individual performs. Each task takes:

- **`description`** — what needs to be done (again using the `{topic}` placeholder)
- **`expected_output`** — what a successful result looks like

Notice we don't manually assign which agent does which task — under `Process.hierarchical` (set up next), the **manager agent decides that at runtime**.

In [8]:
web_research_task = Task(
    description=(
        "Conduct web-based research to identify 5-7 key facts, developments, or statistics about {topic}. "
        "Focus on key use cases, notable companies, and recent growth indicators."
    ),
    expected_output=(
        "A structured list of 5-7 researched facts about {topic}, each with a brief supporting detail."
    ),
)

trend_analysis_task = Task(
    description=(
        "Analyze the research findings to identify and rank the most significant trends in {topic}."
    ),
    expected_output=(
        "A table ranking trends by impact, with concise descriptions of each trend."
    ),
)

report_writing_task = Task(
    description=(
        "Draft a report summarizing the findings and analysis of {topic}. Include sections for "
        "Introduction, Trends Overview, Analysis, and Recommendations."
    ),
    expected_output=(
        "A structured, professional draft with a clear flow of information. Ensure logical organization and consistent tone."
    ),
)

proofreading_task = Task(
    description=(
        "Refine the draft for grammatical accuracy, coherence, and formatting. Ensure the final document is polished "
        "and ready for publication."
    ),
    expected_output=(
        "A professional, polished report free of grammatical errors and inconsistencies. Format the document for "
        "easy readability."
    ),
)

## 7. Assemble the crew

This is the orchestration layer — the **`Crew`** object ties agents, tasks, and process together:

- **`agents`** — the four specialist agents (the manager is passed separately, not in this list)
- **`tasks`** — all four tasks, available for the manager to assign
- **`process=Process.hierarchical`** — this is what makes it a *supervisor* architecture: the manager agent decides task ordering and delegation, rather than tasks always running in the list order
- **`manager_agent`** — the agent given supervisory authority over the others

In [9]:
crew = Crew(
    agents=[web_researcher_agent, trend_analyst_agent, report_writer_agent, proofreader_agent],
    tasks=[web_research_task, trend_analysis_task, report_writing_task, proofreading_task],
    process=Process.hierarchical,
    manager_agent=manager_agent,
    verbose=True,
)

print("✅ Crew assembled — 4 specialist agents + 1 manager, hierarchical process")

✅ Crew assembled — 4 specialist agents + 1 manager, hierarchical process


## 8. Run it — Bangalore's tech industry and startup ecosystem

This is where everything we built actually executes. The manager agent will:

1. Decide which agent should run the web research task first
2. Pass that agent's output to the trend analyst
3. Pass the analysis to the report writer
4. Pass the draft to the proofreader
5. Return the final, polished report

Every `{topic}` placeholder in every agent and task above gets filled in with the same string here — that's the entire mechanism that lets you reuse this whole system for a completely different subject without touching any agent or task code.

In [ ]:
crew_output = crew.kickoff(inputs={"topic": "Bangalore's tech industry and startup ecosystem"})

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e6a7c07f-7672-46eb-9f68-602fcf66ab89                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Workflow Maestro                                                                                        │
│                                                                                                                 │
│  Task: Conduct web-based research to identify 5-7 key facts, developments, or statistics about Bangalore's      │
│  tech industry and startup ecosystem. Focus on key use cases, notable companies, and recent growth indicators.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Research Specialist                                                                                 │
│                                                                                                                 │
│  Task: Conduct web-based research to identify 5-7 key facts, developments, or statistics about Bangalore's      │
│  tech industry and startup ecosystem. Focus on key use cases, notable companies, and recent growth indicators.  │
│  Find complete information about each key point, including brief supporting details.                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 6a463c7a-7018-41e9-8df7-a9aeddd984b1                                                                     │
│  Agent: Workflow Maestro                                                                                        │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Workflow Maestro                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Bangalore Industry Report: Navigating the Growth and Opportunities**                                         │
│                                                                                                                 │
│  **Introduction**                                                                                               │
│                                                                                                                 │
│  Bangalore, often hailed as the Silicon Valley of India, stands as a testament to innovation and progressive    │
│  growth. Known for its vibrant tech ecosystem, the city has mastered the art of blending traditional            │
│  industries with emerging technological sectors. This report delves into the current position of Bangalore      │
│  within the global industrial landscape, highlighting significant trends, analyzing critical factors, and       │
│  presenting strategic recommendations for stakeholders aiming to harness the city's potential.                  │
│                                                                                                                 │
│  **Trends Overview**                                                                                            │
│                                                                                                                 │
│  1. **Investment Influx**: Over the past few years, Bangalore has experienced a remarkable influx of            │
│  investments. Favorable government policies and a burgeoning startup culture have made it a magnet for both     │
│  domestic and foreign investors. This influx has catalyzed a range of industries, with technology, biotech,     │
│  and fintech sectors experiencing significant growth.                                                           │
│                                                                                                                 │
│  2. **Sector Diversification**: While technology continues to be the backbone of Bangalore's economy, there's   │
│  been an observable shift towards sector diversification. Industries such as healthcare, renewable energy, and  │
│  education technology have begun to thrive, attracting talent and investment. This diversification not only     │
│  showcases resilience but also positions Bangalore as a versatile economic hub.                                 │
│                                                                                                                 │
│  3. **Global Recognition**: Bangalore’s evolutionary journey has garnered significant global attention. The     │
│  city has been recognized for its sustainable growth practices, innovation-driven environment, and              │
│  contribution to global technology trends. Such recognition has elevated its status, making it an attractive    │
│  destination for international collaborations and partnerships.                                                 │
│                                                                                                                 │
│  4. **Gender Parity**: Efforts toward achieving gender parity have been commendable. With initiatives           │
│  fostering women's participation in the workforce, particularly in tech sectors, Bangalore is seen as a         │
│  progressive city championing equality. This not only e

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e6a7c07f-7672-46eb-9f68-602fcf66ab89                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: **Bangalore Industry Report: Navigating the Growth and Opportunities**                           │
│                                                                                                                 │
│  **Introduction**                                                                                               │
│                                                                                                                 │
│  Bangalore, often hailed as the Silicon Valley of India, stands as a testament to innovation and progressive    │
│  growth. Known for its vibrant tech ecosystem, the city has mastered the art of blending traditional            │
│  industries with emerging technological sectors. This report delves into the current position of Bangalore      │
│  within the global industrial landscape, highlighting significant trends, analyzing critical factors, and       │
│  presenting strategic recommendations for stakeholders aiming to harness the city's potential.                  │
│                                                                                                                 │
│  **Trends Overview**                                                                                            │
│                                                                                                                 │
│  1. **Investment Influx**: Over the past few years, Bangalore has experienced a remarkable influx of            │
│  investments. Favorable government policies and a burgeoning startup culture have made it a magnet for both     │
│  domestic and foreign investors. This influx has catalyzed a range of industries, with technology, biotech,     │
│  and fintech sectors experiencing significant growth.                                                           │
│                                                                                                                 │
│  2. **Sector Diversification**: While technology continues to be the backbone of Bangalore's economy, there's   │
│  been an observable shift towards sector diversification. Industries such as healthcare, renewable energy, and  │
│  education technology have begun to thrive, attracting talent and investment. This diversification not only     │
│  showcases resilience but also positions Bangalore as a versatile economic hub.                                 │
│                                                                                                                 │
│  3. **Global Recognition**: Bangalore’s evolutionary journey has garnered significant global attention. The     │
│  city has been recognized for its sustainable growth practices, innovation-driven environment, and              │
│  contribution to global technology trends. Such recognition has elevated its status, making it an attractive    │
│  destination for international collaborations and partnerships.                                                 │
│                                                                                                                 │
│  4. **Gender Parity**: Efforts toward achieving gender parity have been commendable. With initiatives           │
│  fostering women's participation in the workforce, par

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



┌───────────────────────────── Execution Traces ──────────────────────────────┐
│                                                                             │
│  🔍 Detailed execution traces are available!                                │
│                                                                             │
│  View insights including:                                                   │
│    • Agent decision-making process                                          │
│    • Task execution flow and timing                                         │
│    • Tool usage details                                                     │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
Would you like to view your execution traces? [y/N] (20s timeout): 

┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                 

## 9. Inspect the result

`crew.kickoff(...)` returns a `CrewOutput` object. The final polished report (the proofreader's output) is what you'll typically want, but the object also gives you access to each individual task's output if you want to inspect the pipeline stage by stage — useful for debugging which agent introduced an issue.

In [11]:
print("===== FINAL REPORT =====\n")
print(crew_output.raw)

===== FINAL REPORT =====

**Bangalore Industry Report: Navigating the Growth and Opportunities**

**Introduction**

Bangalore, often hailed as the Silicon Valley of India, stands as a testament to innovation and progressive growth. Known for its vibrant tech ecosystem, the city has mastered the art of blending traditional industries with emerging technological sectors. This report delves into the current position of Bangalore within the global industrial landscape, highlighting significant trends, analyzing critical factors, and presenting strategic recommendations for stakeholders aiming to harness the city's potential.

**Trends Overview**

1. **Investment Influx**: Over the past few years, Bangalore has experienced a remarkable influx of investments. Favorable government policies and a burgeoning startup culture have made it a magnet for both domestic and foreign investors. This influx has catalyzed a range of industries, with technology, biotech, and fintech sectors experiencing si

In [12]:
# Inspect each stage of the pipeline individually
for i, task_output in enumerate(crew_output.tasks_output, start=1):
    print(f"\n--- Stage {i}: {task_output.agent} ---")
    print(task_output.raw[:500], "..." if len(task_output.raw) > 500 else "")


--- Stage 1: Workflow Maestro ---
1. **Bengaluru-Karnataka's Leading Position:**
   - Bengaluru is ranked as the 15th leading startup ecosystem globally as of 2023, with a significant valuation of over $153 billion. This ranking puts it ahead of global cities like Paris, Tokyo, and Boston. The city is especially strong in domains such as AI and R&D, making it a pivotal tech hub in Asia (Source: https://www.bengalurutechsummit.com/ and GSER 2026).

2. **Notable Companies and Growth:**
   - Bengaluru is home to more than 67,000 te ...

--- Stage 2: Workflow Maestro ---
| Rank | Trend                              | Description                                                                                                                         |
|------|------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------|
| 1    | Booming Investment                 | The influx of ventu

## 10. What just happened — and why this is a *multi-agent* system, not four separate calls

The key thing that makes this different from just calling an LLM four times in a row:

- **You never wrote the sequencing logic.** The manager agent decided which specialist handled which task and in what order — that's the hierarchical `Process` doing real coordination work, not a hardcoded `for` loop.
- **Each agent only does one job, well.** The web researcher never tries to write prose; the proofreader never tries to invent facts. Specialization is enforced by giving each agent a narrow `role`/`goal`/`backstory`, not by a giant single prompt trying to do everything at once.
- **Only one agent has tool access.** `SearchTool` is wired to the web researcher alone — the same principle as giving `search_flights` only to the relevant node in the LangGraph travel-agent examples, just expressed through CrewAI's agent-level tool assignment instead of a graph node.

**Compare to the LangGraph examples earlier in this curriculum:**

| | LangGraph (ReAct / travel agent) | CrewAI (this notebook) |
|---|---|---|
| Orchestration unit | Nodes + edges you wire explicitly | Agents + tasks; a manager agent decides routing |
| Who decides "what's next" | Your `router()` / `decide()` function | The manager agent's own LLM reasoning |
| Control flow visibility | Explicit graph (you can draw it) | Implicit — delegated to the manager at runtime |
| Best fit | When you want precise, inspectable control flow | When the task decomposition itself is something you're comfortable letting an LLM reason about |

**Where to go from here**, per the original article's own suggestion: try a different agent combination, a different architecture (network instead of supervisor), or swap in a different tool — the same way the LangGraph travel-agent notebooks evolved from V1 to V3 to a shipped product.